In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib
import numpy as np

file = '/Users/alejandrogomez-paz/Desktop/UFC Project/3. models/logistic_model/features.csv'
df = pd.read_csv(file, index_col=0)
df['date'] = pd.to_datetime(df['date'])

feature_cols = [c for c in df.columns if c.endswith('_diff')]
df = df.dropna(subset=feature_cols)          # drops debut fights (~27%)

def fit_pipeline(data):
    sc = StandardScaler().fit(data[feature_cols])
    model = LogisticRegression(solver='saga', l1_ratio=1.0, C=0.1, max_iter=5000)
    model.fit(sc.transform(data[feature_cols]), data['y'])
    return sc, model

scaler_full, model_full = fit_pipeline(df)   # main model, every fight

num = 1_000
ensemble = [fit_pipeline(df.sample(len(df), replace=True, random_state=n))
            for n in range(num)]

folder = '/Users/alejandrogomez-paz/Desktop/UFC Project/3. models/logistic_model/'
joblib.dump({'scaler': scaler_full, 'model': model_full,
             'ensemble': ensemble, 'feature_cols': feature_cols},
            folder + 'model.joblib')

# sanity check: bootstrap CI for one example fight
x = df[feature_cols].iloc[[-100]]
ps = np.array([m.predict_proba(sc.transform(x))[0, 1] for sc, m in ensemble])
print(f"saved {num}-model ensemble -> model.joblib")
print(f"example: p={np.median(ps):.3f}, 95% CI [{np.percentile(ps, 2.5):.3f}, {np.percentile(ps, 97.5):.3f}]")

saved 1000-model ensemble -> model.joblib
example: p=0.373, 95% CI [0.328, 0.418]
